# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook demonstrates how to use the `mlcroissant` library to explore and analyze the FAIR² dataset: **Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells**.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata object
ds = mlc.Dataset(croissant_url)
print(f"Loaded dataset: {ds.metadata.name}\n\nDescription: {ds.metadata.description}")

## 2. Data Overview
Review available Record Sets, their `@id` values, and the fields/columns available in the dataset.

Entities are referenced by their `@id` property per Croissant specification. This ensures accurate mapping to the data structure.

In [ ]:
# List all available record sets and their fields
from pprint import pprint

record_sets = ds.metadata.record_sets
print(f"There are {len(record_sets)} record set(s) in the dataset:\n")
for rs in record_sets:
    print(f"- Record Set Name: {rs.name}")
    print(f"  @id: {rs.id}")
    if rs.fields:
        print("  Fields and their @id values:")
        for field in rs.fields:
            print(f"    - {field.name}: {field.id}")
    print("")

Below is a preview of the records from the main record set (using its `@id`). Replace `<record_set_id>` with an actual value as discovered above.

In [ ]:
# Preview sample records from the main record set
# Identify the primary table-like record set (usually only one for tabular data)
tabular_rs = None
for rs in record_sets:
    if rs.fields and len(rs.fields) > 3:  # Simple heuristic for the main table
        tabular_rs = rs
        break
if tabular_rs is None:
    tabular_rs = record_sets[0]
print(f"Previewing records from Record Set: {tabular_rs.name} [@id: {tabular_rs.id}]\n")
for idx, rec in enumerate(ds.records(record_set=tabular_rs.id)):
    pprint(rec)
    if idx >= 2:
        break

## 3. Data Extraction
Load data from each record set into a DataFrame for analysis, referencing all entities by their `@id`. 
You can select which record set(s) to analyze further. For this notebook, we'll focus on the main table above.

In [ ]:
# Extract records from all record sets into pandas DataFrames
dataframes = {}
for rs in record_sets:
    records = list(ds.records(record_set=rs.id))
    if records:  # Only create DataFrame if there is data
        dataframes[rs.id] = pd.DataFrame(records)
        print(f"Loaded DataFrame for Record Set: {rs.name} [@id: {rs.id}] with shape {dataframes[rs.id].shape}")
    else:
        print(f"No records found for {rs.name} [@id: {rs.id}]")

# For demonstration, we'll continue using the primary tabular record set
main_rs_id = tabular_rs.id
print("\nColumns in the main table:", dataframes[main_rs_id].columns.tolist())
dataframes[main_rs_id].head()

## 4. Exploratory Data Analysis (EDA)
Apply data processing steps—such as filtering records, normalizing numeric fields, and grouping/categorizing—referencing fields by their `@id`.

Inspect which numeric fields are available (by `@id`) and select one for filtering/normalization.

In [ ]:
# List numeric and categorical fields by their @id
numeric_field_id = None
group_field_id = None

for field in tabular_rs.fields:
    # Common numeric types: Integer, Float, Number
    if hasattr(field, 'data_type') and (field.data_type in ['schema:Integer','schema:Float','schema:Number','Integer','Float','Number']):
        print(f"Numeric Field: {field.name} [@id: {field.id}, data_type: {field.data_type}]")
        if not numeric_field_id:
            numeric_field_id = field.id
    # Pick first categorical field
    if group_field_id is None and hasattr(field, 'data_type') and field.data_type == 'schema:Text':
        group_field_id = field.id

print(f"\nChosen numeric field for EDA: {numeric_field_id}")
if group_field_id:
    print(f"And chosen group (categorical) field: {group_field_id}")

df = dataframes[main_rs_id].copy()
if numeric_field_id not in df.columns:
    raise ValueError(f"Field '{numeric_field_id}' not in DataFrame columns: {df.columns.tolist()}")

# Filter for values greater than a threshold
threshold = df[numeric_field_id].mean() if df[numeric_field_id].dtype.kind in 'fi' else 10
filtered_df = df[df[numeric_field_id] > threshold]
print(f"Filtered records with {numeric_field_id} > {threshold} (mean): {filtered_df.shape[0]} rows")
print(filtered_df.head(3))

# Normalize the numeric field (z-score)
filtered_df[numeric_field_id + "_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
print(filtered_df[[numeric_field_id, numeric_field_id + "_normalized"]].head(3))

# Group by a categorical field, if present
if group_field_id in filtered_df.columns:
    grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
    print(f"Grouped mean {numeric_field_id} by {group_field_id} (first 5 groups):")
    print(grouped_df.head())

## 5. Visualization
Visualize the distribution of the selected numeric field and its normalized version, and (optionally) a grouped summary.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(9,5))
sns.histplot(df[numeric_field_id].dropna(), kde=True, bins=30)
plt.title(f"Distribution of {numeric_field_id}")
plt.xlabel(numeric_field_id)
plt.ylabel("Count")
plt.show()

if group_field_id in df.columns:
    plt.figure(figsize=(12,5))
    sns.boxplot(data=df, x=group_field_id, y=numeric_field_id)
    plt.title(f"{numeric_field_id} by {group_field_id}")
    plt.xticks(rotation=45)
    plt.show()

## 6. Conclusion
- We loaded the dataset and schema using `mlcroissant`, referencing all entities by their `@id`.
- The dataset provides a rich table of protein characteristics, easily accessible for analysis.
- We demonstrated basic filtering and normalization on a chosen numeric field, and grouping/categorization if applicable.
- Further domain-specific investigation (such as functional analysis or advanced visualizations) can be performed according to research needs.

For more information, see the dataset's [Croissant schema](https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json) or use `mlcroissant` API for advanced access.